# Agentic search with BrowseComp-Plus


[**Open in Colab**](https://colab.research.google.com/github/HassanAlgoz/B5/blob/main/content/W4/M4/lessons/02_agentic_search.ipynb)


| Step | Tech | Execution |
| --- | --- | --- |
| Vector store | ChromaDB | 💻 Local |
| LLM | OpenRouter | 🌐 Remote |

## Colab Setup


It is recommended to use a GPU-enabled runtime if your environment benefits from faster local embedding work in Chroma; the lesson LLM runs on OpenRouter.


### Install dependencies


In [1]:
%pip install -qqq chromadb datasets langchain langchain-openai langgraph python-dotenv


In [2]:
import json
import os
import re
from itertools import islice

### Environment variables


In [3]:
from dotenv import load_dotenv


def _get_env_from_colab_or_os(key: str) -> str | None:
    """Read an environment variable from Colab secrets first, then OS env."""
    try:
        from google.colab import userdata
        try:
            return userdata.get(key)
        except userdata.SecretNotFoundError:
            pass
    except ImportError:
        pass
    return os.getenv(key)


In [4]:
load_dotenv()


False

## 1. BrowseComp-Plus: streaming data and Chroma index

BrowseComp-Plus extends OpenAI's BrowseComp: hard queries that need several retrieval rounds plus reasoning. Each query has **gold**, **evidence**, and **negative** documents.

We use **`load_dataset(..., streaming=True)`** so rows are iterated on demand (no full local materialization). The Chroma index keeps a **small corpus slice** (default 100 docs) for a fast demo; raise `max_docs` for broader coverage.


### 1.1 Hugging Face streaming configuration


In [ ]:
from datasets import get_dataset_split_names

BC_PLUS_QUERIES = "Tevatron/browsecomp-plus"
BC_PLUS_CORPUS = "Tevatron/browsecomp-plus-corpus"


def _pick_split(path: str) -> str:
    names = list(get_dataset_split_names(path))
    for preferred in ("train", "test", "validation"):
        if preferred in names:
            return preferred
    return names[0]


QUERIES_STREAM_SPLIT = _pick_split(BC_PLUS_QUERIES)
CORPUS_STREAM_SPLIT = _pick_split(BC_PLUS_CORPUS)

print(
    "Streaming splits:",
    f"{BC_PLUS_QUERIES} [{QUERIES_STREAM_SPLIT}]",
    "|",
    f"{BC_PLUS_CORPUS} [{CORPUS_STREAM_SPLIT}]",
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Streaming splits: Tevatron/browsecomp-plus [test] | Tevatron/browsecomp-plus-corpus [train]


### 1.2 Chroma client, helpers, and ingestion


In [6]:
import chromadb

chroma = chromadb.EphemeralClient()

In [7]:
from datasets import load_dataset


def tokenize(text: str) -> set[str]:
    return {t for t in re.findall(r"[a-z0-9]+", text.lower()) if len(t) > 1}


def _pick_first(row: dict, keys: list[str], default=""):
    for k in keys:
        if k in row and row[k] is not None:
            return row[k]
    return default


def build_browsecomp_collection(max_docs: int = 100):
    col = chroma.get_or_create_collection(
        name="browsecomp-plus-corpus",
        metadata={"hnsw:space": "cosine"},
    )
    if col.count() > 0:
        return col

    corpus_stream = load_dataset(
        BC_PLUS_CORPUS,
        split=CORPUS_STREAM_SPLIT,
        streaming=True,
    )
    ids, docs, metas = [], [], []
    for i, row in enumerate(islice(corpus_stream, max_docs)):
        doc_id = str(_pick_first(row, ["doc_id", "id", "document_id"], default=f"doc-{i}"))
        text = _pick_first(row, ["text", "contents", "content", "document"], default="")
        ids.append(f"corpus-{i}")
        docs.append(text)
        metas.append({"doc_id": doc_id, "query": False})

    col.add(ids=ids, documents=docs, metadatas=metas)
    return col

Build the collection

In [8]:
collection = build_browsecomp_collection(max_docs=100)
print("indexed_docs:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 29.6MiB/s]


indexed_docs: 100


Note: Chroma **embeds** on the fly—the stored documents, using its default embedding model (`/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz`).

## 2. Search tools

Each tool returns records `{id, docId, document}`:

- **semantic** (embedding-based)
- **lexical** (keyword overlap on the indexed slice)
- **hybrid** (RRF over both)

### 2.1 Tool implementations


Helper functions to convert list of tuples, to a list of dictionaries:

In [ ]:
CORPUS_FILTER = {"query": {"$ne": True}}


def rows_to_records(rows) -> list[dict]:
    out = []
    for r in rows:
        out.append(
            {
                "id": r["id"],
                "docId": (r.get("metadata") or {}).get("doc_id", "?"),
                "document": r.get("document") or "",
            }
        )
    return out

Semantic search function, calls `collection.query(...)` with proper arguments:

In [9]:
def semantic_search(collection, query: str, n: int = 5) -> list[dict]:
    res = collection.query(
        query_texts=[query],
        n_results=n,
        where=CORPUS_FILTER,
    )
    rows = [{"id": i, "metadata": m, "document": d} for i, m, d in zip(res["ids"][0], res["metadatas"][0], res["documents"][0])]
    return rows_to_records(rows)

Lexical search function, calls `collection.get(...)`, then sorts the docuemnts:

In [ ]:
def lexical_search(collection, query: str, n: int = 5) -> list[dict]:
    q_tokens = tokenize(query)
    res = collection.get(where=CORPUS_FILTER, include=["documents", "metadatas"])
    scored = []
    for i, doc in enumerate(res["documents"]):
        score = len(q_tokens & tokenize(doc or ""))
        scored.append((score, i))
    scored.sort(key=lambda x: (-x[0], x[1]))
    rows = []
    for _, idx in scored[:n]:
        rows.append(
            {
                "id": res["ids"][idx],
                "metadata": res["metadatas"][idx],
                "document": res["documents"][idx],
            }
        )
    return rows_to_records(rows)


- The `rrf_merge` function is used in `hybrid_search` to factor in scores coming from different scales (lexical and semantic)
- Notice how `hybrid_search` uses both `semantic_search` and `lexical_search` functions followed by `rrf_merge`

In [10]:
def rrf_merge(
    ranked_lists: list[list[str]],
    weights: list[float],
    k: int = 60,
    limit: int = 5,
) -> list[str]:
    scores: dict[str, float] = {}
    for ranks, w in zip(ranked_lists, weights):
        for r, doc_id in enumerate(ranks):
            scores[doc_id] = scores.get(doc_id, 0.0) + w / (k + r + 1)
    ordered = sorted(scores.keys(), key=lambda x: -scores[x])
    return ordered[:limit]


def hybrid_search(
    collection,
    dense_query: str,
    sparse_query: str,
    n: int = 5,
    dense_weight: float = 2.0,
    sparse_weight: float = 1.0,
) -> list[dict]:
    sem = semantic_search(collection, dense_query, n=20)
    lex = lexical_search(collection, sparse_query, n=20)
    merged_ids = rrf_merge(
        [[r["id"] for r in sem], [r["id"] for r in lex]],
        [dense_weight, sparse_weight],
        limit=n,
    )
    by_id = {r["id"]: r for r in sem + lex}
    return [by_id[i] for i in merged_ids if i in by_id]


Now we define these as tools for LangChain. We first need a `format_tool_result` function to structure the retrieved documents in the LLM nicely:

In [11]:
from langchain.tools import tool


def format_tool_result(records: list[dict]) -> str:
    if not records:
        return "No records found"
    parts = []
    for rec in records:
        parts.append(f"// Document {rec['id']}\n// Corpus document ID: {rec['docId']}\n{rec['document']}")
    return "\n\n".join(parts)


@tool
def semantic_search_tool(query: str, num_results: int = 5, reason: str = "") -> str:
    """Dense-vector semantic search. Use for conceptually related passages."""
    recs = semantic_search(collection, query, n=num_results)
    return format_tool_result(recs)


@tool
def lexical_search_tool(query: str, num_results: int = 5, reason: str = "") -> str:
    """Keyword-style search for exact terms, names, and numbers."""
    recs = lexical_search(collection, query, n=num_results)
    return format_tool_result(recs)


@tool
def hybrid_search_tool(
    dense_query: str,
    sparse_query: str,
    num_results: int = 5,
    dense_weight: float = 2.0,
    sparse_weight: float = 1.0,
    reason: str = "",
) -> str:
    """Semantic + lexical via RRF. Use for constraint-heavy questions."""
    recs = hybrid_search(
        collection,
        dense_query,
        sparse_query,
        n=num_results,
        dense_weight=dense_weight,
        sparse_weight=sparse_weight,
    )
    return format_tool_result(recs)


SEARCH_TOOLS = [semantic_search_tool, lexical_search_tool, hybrid_search_tool]

## 3. LangGraph workflow and LangChain agent

The functional API uses **`@task`** for one unit of work per LLM or agent run and **`@entrypoint`** for the outer plan loop: plan, execute each step with a tool-calling agent, structure outcomes, evaluate the plan, then produce a grounded final answer. **`create_agent`** owns the ReAct tool loop for each step.


### 3.1 LLM (OpenRouter)


In [12]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free — tool calling, free tier
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=_get_env_from_colab_or_os("OPENROUTER_API_KEY"),
)
llm = model_nemotron3_nano


TimeoutException: Requesting secret OPENROUTER_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

### 3.2 Schemas, prompts, and helpers


In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class PlanStep(BaseModel):
    id: str
    title: str
    description: str = ""
    parents: list[str] = Field(default_factory=list)


class PlanOut(BaseModel):
    steps: list[PlanStep]


class StepOutcome(BaseModel):
    stepId: str
    summary: str
    evidence: list[str]
    candidateAnswers: list[str] | None = None


class PlanEval(BaseModel):
    decision: Literal["continue", "finalize", "overridePlan"]
    reason: str
    newSteps: list[PlanStep] | None = None


class FinalAnswer(BaseModel):
    answer: str
    reason: str
    evidence: list[str]
    confidence: float = Field(ge=0.0, le=1.0)


PLAN_SYSTEM = """You are a query planner for a multi-step search agent over a fixed corpus.
Produce steps (id, title, description, optional parents). Do not answer the question. Do not invent document ids."""

EXEC_SYSTEM = """You execute ONE step of a search plan using tools. Call tools until you have enough evidence, then respond with a brief summary (no more tool calls)."""

EVAL_SYSTEM = """Given the question, plan, and history, choose:
- continue: run remaining steps
- finalize: stop planning; enough evidence for final answer
- overridePlan: replace pending steps with new ones (provide newSteps)."""

FINAL_SYSTEM = """Produce the final answer from evidence only. Ground in cited chunk ids."""


def step_user_block(user_query: str, plan: list[dict], step: dict, history: list[dict]) -> str:
    lines = [
        f"Original question: {user_query}",
        "Plan:",
        *[f"  {s['id']}: {s['title']}" for s in plan],
        f"Current step: {step['id']}: {step['title']}\n{step.get('description', '')}",
    ]
    if history:
        lines.append("Prior outcomes:")
        for h in history:
            lines.append(json.dumps(h, indent=2))
    return "\n".join(lines)


def serialize_messages(msgs) -> list[dict]:
    out = []
    for m in msgs:
        role = getattr(m, "type", None) or ""
        content = m.content
        if not isinstance(content, str):
            content = str(content)
        out.append({"role": role, "content": content})
    return out


### 3.3 Retrieval agent (`create_agent`)


In [ ]:
from langchain.agents import create_agent

retrieval_agent = create_agent(
    model=model_nemotron3_nano,
    tools=SEARCH_TOOLS,
    system_prompt=EXEC_SYSTEM,
)


### 3.4 Tasks and workflow entrypoint


In [ ]:
from langchain_core.utils.uuid import uuid7
from langgraph.func import entrypoint, task


@task
def plan_generation_task(inp: dict) -> dict:
    user_query = inp["user_query"]
    max_plan_size = inp["max_plan_size"]
    planner = llm.with_structured_output(PlanOut)
    out = planner.invoke(
        [
            ("system", PLAN_SYSTEM),
            ("user", f"Max {max_plan_size} steps.\nQuestion: {user_query}"),
        ]
    )
    return {"steps": [s.model_dump() for s in out.steps]}


@task
def retrieval_step_task(ctx: dict) -> dict:
    text = step_user_block(ctx["user_query"], ctx["plan"], ctx["step"], ctx["history"])
    result = retrieval_agent.invoke(
        {"messages": [("user", text)]},
        {"recursion_limit": 40},
    )
    return {"messages": serialize_messages(result["messages"])}


@task
def outcome_task(ctx: dict) -> dict:
    step = ctx["step"]
    transcript = json.dumps(ctx["agent_messages"], indent=2)
    structured = llm.with_structured_output(StepOutcome).invoke(
        [
            (
                "system",
                "Finalize the search step. Evidence must be Chroma record ids from // Document lines.",
            ),
            (
                "user",
                f"Step {step['id']}: {step['title']}\nTranscript:\n{transcript}",
            ),
        ]
    )
    return structured.model_dump()


@task
def evaluate_plan_task(inp: dict) -> dict:
    structured = llm.with_structured_output(PlanEval).invoke(
        [
            ("system", EVAL_SYSTEM),
            (
                "user",
                json.dumps(
                    {
                        "question": inp["user_query"],
                        "plan": inp["plan"],
                        "history": inp["history"],
                    }
                ),
            ),
        ]
    )
    return structured.model_dump()


@task
def final_answer_task(inp: dict) -> dict:
    structured = llm.with_structured_output(FinalAnswer).invoke(
        [
            ("system", FINAL_SYSTEM),
            (
                "user",
                json.dumps({"question": inp["user_query"], "history": inp["history"]}),
            ),
        ]
    )
    return structured.model_dump()

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def agentic_workflow(inp: dict) -> dict:
    user_query = inp["user_query"]
    max_plan_size = inp.get("max_plan_size", 4)
    raw = plan_generation_task({"user_query": user_query, "max_plan_size": max_plan_size}).result()
    plan = list(raw["steps"])
    history: list[dict] = []
    idx = 0
    while idx < len(plan):
        step = plan[idx]
        ctx = {
            "user_query": user_query,
            "plan": plan,
            "step": step,
            "history": history,
        }
        ar = retrieval_step_task(ctx).result()
        oc = outcome_task({**ctx, "agent_messages": ar["messages"]}).result()
        history.append(oc)
        ev = evaluate_plan_task(
            {"user_query": user_query, "plan": plan, "history": history}
        ).result()
        if ev.get("decision") == "finalize":
            break
        if ev.get("decision") == "overridePlan" and ev.get("newSteps"):
            plan = plan[: idx + 1] + list(ev["newSteps"])
        idx += 1
    final = final_answer_task({"user_query": user_query, "history": history}).result()
    return {"plan": plan, "history": history, "final": final}


## 4. Running the workflow


Use the same `config` (`thread_id`) for `invoke` or `stream` so checkpointing stays consistent. Increase `recursion_limit` in `retrieval_agent.invoke` if a step needs extra tool rounds.


In [ ]:
# for chunk in agentic_workflow.stream(
#     {"user_query": "What causes an aurora?"},
#     {"configurable": {"thread_id": str(uuid7())}},
# ):
#     print(chunk)

In [15]:
def get_query_text_by_id(query_id: int | str) -> str:
    qid = str(query_id)
    stream = load_dataset(
        BC_PLUS_QUERIES,
        split=QUERIES_STREAM_SPLIT,
        streaming=True,
    )
    for row in stream:
        row_qid = str(_pick_first(row, ["query_id", "id", "qid"], default=""))
        if row_qid == qid:
            return _pick_first(row, ["query", "question", "text"], default="")
    return ""


query_770 = get_query_text_by_id(770)
print("Loaded query 770:\n", query_770)

_cfg = {"configurable": {"thread_id": str(uuid7())}}
result = agentic_workflow.invoke(
    {"user_query": query_770, "max_plan_size": 6},
    _cfg,
)
print(json.dumps(result, indent=2))


Loaded query 770:
 Y0cJn+rqYcZgG83ntcd/sbISycE6uZkhaPJsvZiT1lpFCBWd6qNuwHFO3Pn6xn667RKdiXK5tjMl+CryutbBV01KGYGu+CibJhed4bLUNry5VtTfNv2CIWm3O7ONk9ZaRQgfnOG4fMB7Wsn6qJF5s/dTnds66pIhd/Qk8pnBzUdQCBqc+6R8zHEb1Pv6gybl7hydiXK5tC8o8ii7itbGEkEIHpzhoTjZYFnR/KnZc7H3W9OJbanGeCX1NfKs3NdGTE0YlOvkOIk4G+n9v5F/u7Nby8A77JYsJeAlppaT1VpPRVyH5q9hiXZUkPC+2GKwsxLJwTq5lS9q/Gyln8CCUwBDGYrgpWzMNUjN8Lvac6f3U8mJPrmUL2vxKaCb3cFXAEES07z6KZA7G5C1idRko7JWncgsuYMoYLcvvZDFx1xPWlyc6Op5iWVa0/C2kXSwsV3PzH+rx3I1uWz/3uPXUExBD5vrrjjIexvc567YdbmyEtTHf6vHcTe5bP/e8M1fUEQZh+uuON19XtTn+uF+kfdd04kr8ZJgcuUlppfdxUEARxrT76Q47Htc0fyp2TaipVvJzC23


NameError: name 'llm' is not defined